# SDXL y Qwen3-VL · Generación y análisis visual

Prueba para PlanIA P2. SDXL 1.0 Base genera un retrato a partir de la foto de Liam en **cuda:0**. Qwen3-VL-4B-Instruct analiza el resultado y lo compara con la entrada en **cuda:1**. Ambos modelos ejecutan la inferencia en Kaggle.

Abre **Edit**, selecciona **GPU T4 ×2** en **Accelerator**, activa **Internet** para descargar los pesos y pulsa **Run All**. La foto de Liam ya está adjunta. Puedes cambiar la imagen y el prompt en el paso 2.

[Dataset de Liam](https://www.kaggle.com/datasets/jeffreyamc/plania-p2-liam-input) · [SDXL 1.0 Base](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) · [Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct)

## 1. Instalar dependencias

Usamos el PyTorch con CUDA que ya incluye Kaggle y fijamos las versiones de las librerías de inferencia.

In [ ]:
%pip install -q diffusers==0.35.2 transformers==4.57.1 accelerate==1.11.0

## 2. Elegir la imagen y el prompt

Con **image_path = None**, el notebook busca **liam.jpeg** en el dataset adjunto. Para usar otra foto, adjúntala con **Add Input** y copia su ruta bajo **/kaggle/input/** en **image_path**.

El prompt describe el estilo y los rasgos que se quieren conservar. **strength = 0.50** limita cuánto se transforma la entrada. Puedes bajar ese valor para conservar más rasgos o subirlo para acentuar la estilización.

In [ ]:
image_path = None  # Usa liam.jpeg del dataset adjunto.
prompt = (
    "A realistic 3D animated-film portrait of the same cat in the reference photo. "
    "Preserve its facial proportions, pale blue eyes, pink nose, cream-white fur and "
    "light orange markings around the eyes and ears. Keep the same pose and expression, "
    "with softly rendered fur and natural lighting."
)
strength = 0.50
num_inference_steps = 25
guidance_scale = 7.5
seed = 42

## 3. Cargar la foto y comprobar la GPU

Kaggle monta el dataset bajo **/kaggle/input/**. Respetamos la orientación de la foto y la ajustamos a **768 × 1024**, que mantiene las proporciones de Liam. Si usas otra imagen, añadimos relleno cuando sea necesario. Comprobamos que haya dos GPUs y reservamos la primera para SDXL.

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from IPython.display import display
from PIL import Image, ImageOps

if torch.cuda.device_count() < 2:
    raise RuntimeError("Selecciona GPU T4 ×2 en Accelerator y reinicia la sesión.")

device = torch.device("cuda:0")
gpu = torch.cuda.get_device_properties(device)
gib = 1024**3
print(f"GPU: {gpu.name} | VRAM visible para PyTorch: {gpu.total_memory / gib:.2f} GiB")

if image_path is None:
    image_path = next(Path("/kaggle/input").glob("**/plania-p2-liam-input/liam.jpeg"), None)
    if image_path is None:
        raise FileNotFoundError("Adjunta el dataset plania-p2-liam-input o indica image_path.")

with Image.open(image_path) as source:
    input_image = ImageOps.exif_transpose(source).convert("RGB")

input_image = ImageOps.pad(
    input_image, (768, 1024), Image.Resampling.LANCZOS, color="#f8f2e7"
)

output_dir = Path("/kaggle/working")
input_image.save(output_dir / "entrada.png")

## 4. Cargar SDXL 1.0 Base

Descargamos una revisión fija del modelo y cargamos sus pesos en la primera GPU en FP16. El modelo base genera la imagen por sí solo. La primera descarga puede tardar varios minutos.

In [ ]:
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
model_revision = "462165984030d82259a11f4367a4eed129e94a7b"

torch.cuda.reset_peak_memory_stats(device)
load_start = perf_counter()
pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    model_id,
    revision=model_revision,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipeline.to(device)
pipeline.set_progress_bar_config(disable=True)
torch.cuda.synchronize(device)
load_seconds = perf_counter() - load_start

## 5. Generar y guardar una variación

La imagen y el prompt guían la generación. Guardamos la entrada preparada, la variación y un JSON con los parámetros, el tiempo y el pico de memoria de PyTorch durante la carga y la generación. Al final del notebook, el JSON también incluirá las mediciones de Qwen.

En una sesión interactiva encontrarás los archivos en **/kaggle/working**. Para conservarlos en la pestaña **Output**, usa **Save Version → Save & Run All**.

In [ ]:
generator = torch.Generator(device=device).manual_seed(seed)

generation_start = perf_counter()
result = pipeline(
    prompt=prompt,
    image=input_image,
    strength=strength,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator,
)
torch.cuda.synchronize(device)
generation_seconds = perf_counter() - generation_start

generated_image = result.images[0]
generated_image.save(output_dir / "generada.png")

summary = {
    "status": "running",
    "stage": "sdxl_complete",
    "model": model_id,
    "revision": model_revision,
    "dtype": "float16",
    "gpu": gpu.name,
    "device": str(device),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gib": round(gpu.total_memory / gib, 2),
    "vram_peak_allocated_gib": round(torch.cuda.max_memory_allocated(device) / gib, 2),
    "vram_peak_reserved_gib": round(torch.cuda.max_memory_reserved(device) / gib, 2),
    "load_seconds": round(load_seconds, 2),
    "generation_seconds": round(generation_seconds, 2),
    "versions": {name: version(name) for name in ("torch", "diffusers", "transformers", "accelerate")},
    "input_source": str(image_path),
    "size_px": list(generated_image.size),
    "prompt": prompt,
    "strength": strength,
    "num_inference_steps": num_inference_steps,
    "guidance_scale": guidance_scale,
    "seed": seed,
    "artifacts": ["entrada.png", "generada.png", "run_summary.json"],
}
(output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Generación: {generation_seconds:.2f} s | Pico asignado: {summary['vram_peak_allocated_gib']:.2f} GiB")
print("Entrada a la izquierda y variación generada a la derecha.")
preview_size = (288, 384)
preview = Image.new("RGB", (2 * preview_size[0] + 8, preview_size[1]), "white")
preview.paste(input_image.resize(preview_size), (0, 0))
preview.paste(generated_image.resize(preview_size), (preview_size[0] + 8, 0))
display(preview)

## 6. Cargar Qwen3-VL en la segunda GPU

Qwen3-VL-4B-Instruct recibe imágenes y texto. Cargamos sus pesos en FP16 y fijamos el modelo completo en **cuda:1**. SDXL permanece en **cuda:0**.

[API de Qwen3-VL en Transformers 4.57.1](https://huggingface.co/docs/transformers/v4.57.1/model_doc/qwen3_vl)

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

qwen_id = "Qwen/Qwen3-VL-4B-Instruct"
qwen_revision = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
qwen_device = torch.device("cuda:1")
qwen_gpu = torch.cuda.get_device_properties(qwen_device)

torch.cuda.reset_peak_memory_stats(qwen_device)
qwen_load_start = perf_counter()
processor = AutoProcessor.from_pretrained(qwen_id, revision=qwen_revision)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    qwen_id,
    revision=qwen_revision,
    dtype=torch.float16,
    device_map={"": qwen_device.index},
    attn_implementation="sdpa",
).eval()
torch.cuda.synchronize(qwen_device)
qwen_load_seconds = perf_counter() - qwen_load_start

assert pipeline.device == device
assert all(parameter.device == qwen_device for parameter in qwen_model.parameters())
print(f"SDXL: {pipeline.device} | Qwen: {qwen_model.device} ({qwen_gpu.name})")

## 7. Describir el retrato y compararlo con la foto

Le pasamos dos imágenes en orden: primero la entrada y luego el retrato de SDXL. Reducimos ambas para limitar los tokens visuales de Qwen. Las imágenes guardadas conservan su resolución.

Qwen responde en español con observaciones del retrato y cambios frente a la entrada. Guardamos su respuesta en **analisis_qwen.md** y medimos su tiempo y memoria en **run_summary.json**.

In [ ]:
analysis_prompt = (
    "Eres especialista en fotografía y dirección de arte. La primera imagen es "
    "la foto de referencia y la segunda es el retrato generado. Responde en español. "
    "En 'Retrato generado', describe la composición, iluminación, paleta, textura "
    "y estilo de la segunda imagen. En 'Comparación con la referencia', explica "
    "qué rasgos del rostro, colores, marcas y pose se conservan y cuáles cambian. "
    "Señala artefactos solo si son visibles. Fundamenta cada observación en detalles "
    "de las imágenes y expresa cualquier duda. Escribe dos párrafos de hasta 80 palabras cada uno."
)
analysis_images = [
    ImageOps.contain(image, (768, 768), Image.Resampling.LANCZOS)
    for image in (input_image, generated_image)
]
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "image"},
            {"type": "text", "text": analysis_prompt},
        ],
    }
]

qwen_start = perf_counter()
chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[chat], images=analysis_images, return_tensors="pt")
inputs.pop("token_type_ids", None)
inputs = inputs.to(qwen_device, dtype=torch.float16)

with torch.inference_mode():
    output_ids = qwen_model.generate(**inputs, max_new_tokens=640, do_sample=False)
torch.cuda.synchronize(qwen_device)
qwen_seconds = perf_counter() - qwen_start

new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
description = processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
if not description:
    raise RuntimeError("Qwen devolvió un análisis vacío.")

analysis_path = output_dir / "analisis_qwen.md"
analysis_path.write_text(f"# Análisis visual de Qwen3-VL\n\n{description}\n", encoding="utf-8")
summary["qwen"] = {
    "model": qwen_id,
    "revision": qwen_revision,
    "device": str(qwen_device),
    "gpu": qwen_gpu.name,
    "dtype": "float16",
    "vram_total_gib": round(qwen_gpu.total_memory / gib, 2),
    "vram_peak_allocated_gib": round(torch.cuda.max_memory_allocated(qwen_device) / gib, 2),
    "vram_peak_reserved_gib": round(torch.cuda.max_memory_reserved(qwen_device) / gib, 2),
    "load_seconds": round(qwen_load_seconds, 2),
    "analysis_seconds": round(qwen_seconds, 2),
    "image_sizes_px": [list(image.size) for image in analysis_images],
    "generated_tokens": new_ids.shape[1],
}
summary.update(status="complete", stage="done")
summary["artifacts"].append(analysis_path.name)
(output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(description)
print(f"Análisis: {qwen_seconds:.2f} s | Pico asignado: {summary['qwen']['vram_peak_allocated_gib']:.2f} GiB")